# GastroVision — Baseline (shared base notebook)

**AIN501 Final Project · Data 70 / Model 30**

This notebook is the **shared foundation** for the whole team. It establishes:
1. Reproducible config + seeding
2. Dataset loading with a **fixed, published stratified 60:20:20 split** (no ad-hoc re-splitting → no leakage)
3. A **shared evaluation harness** (macro-F1, per-class F1, confusion matrix, multi-seed runner)
4. The **DenseNet-121 baseline** to reproduce (target macro-F1 ~0.65)

> Do NOT change the split / seed / eval cells without telling the team — everyone's
> numbers must be comparable. Members A/B/C branch out from the marked sections at the bottom.

**Target runtime: Google Colab Free (T4).** One run ~15-25 min @224px. Also works on Kaggle.

## 1 · Setup & reproducibility

In [ ]:
# Colab already ships torch/torchvision/sklearn/pandas/matplotlib/seaborn.
# `timm` (needed by Member B for Swin/ViT/ConvNeXt/CoAtNet) is NOT preinstalled:
!pip install -q timm

import os, random, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision as tv
from torchvision import transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch:', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU. On Colab: Runtime > Change runtime type > T4 GPU.')

In [ ]:
def set_seed(seed: int):
    """Full reproducibility. We report mean +/- std over >=3 seeds (rubric rule #4)."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [0, 1, 2]        # >=3 seeds required
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30              # baseline; tune per experiment
NUM_WORKERS = 2
PIN_MEMORY = (DEVICE == 'cuda')   # pinning only helps host->GPU copies; avoids a warning on CPU
set_seed(SEEDS[0])

## 2 · Get the data (auto-detects Colab vs local)

GastroVision must be downloaded from the official source
([GitHub](https://github.com/DebeshJha/GastroVision) → download link / request form).

The next cell **detects the environment** and sets paths accordingly:
- **Colab:** mounts Drive, unzips your `gastrovision.zip` into `/content`, and saves checkpoints
  to Drive (so they survive disconnects).
- **Local:** reads the dataset from the repo's `data/gastrovision/` folder and writes checkpoints
  to `checkpoints/` (both git-ignored).

The compute device (GPU/CPU) is already auto-detected in section 1 — no change needed to run either place.

In [ ]:
# --- Environment-aware paths + auto-download: runs on Colab AND locally. ---
import subprocess

try:
    from google.colab import drive
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False
print('environment:', 'Colab' if IS_COLAB else 'local')

# Official public dataset folder (Google Drive), from https://github.com/DebeshJha/GastroVision
GDRIVE_FOLDER = 'https://drive.google.com/drive/folders/1oT9Vez7pfhrN44Korx6wHRBAGwyHc7Cb'

if IS_COLAB:
    DATA_DIR   = Path('/content/gastrovision')
    OUTPUT_DIR = Path('/content/outputs')
    drive.mount('/content/drive')                                  # for checkpoint persistence
    CKPT_DIR   = Path('/content/drive/MyDrive/gastrovision_ckpts')
    # Auto-download the dataset with gdown (no manual upload). First run only.
    if not DATA_DIR.exists():
        subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
        subprocess.run(['gdown', '--folder', '--remaining-ok',
                        '-O', str(DATA_DIR), GDRIVE_FOLDER], check=True)
    # If Drive-folder download gets rate-limited, fall back to a zip on your Drive:
    #   subprocess.run(['unzip','-q','-n','/content/drive/MyDrive/gastrovision.zip','-d','/content/'])
else:
    REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_DIR   = REPO / 'data' / 'gastrovision'   # TODO: put/symlink the dataset here
    CKPT_DIR   = REPO / 'checkpoints'
    OUTPUT_DIR = REPO / 'outputs'
    # Local download (optional):
    #   pip install gdown && gdown --folder --remaining-ok -O <DATA_DIR> <GDRIVE_FOLDER>

CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# gdown may create a nested folder (e.g. .../gastrovision/GastroVision/...). The class-count
# sanity check in the next code cell (NUM_CLASSES == 22) will flag a wrong DATA_DIR level.
assert DATA_DIR.exists(), (
    f'Dataset not found at {DATA_DIR}. '
    'Colab: check gdown output. Local: place the dataset under data/gastrovision/.')
print('DATA_DIR   =', DATA_DIR)
print('CKPT_DIR   =', CKPT_DIR)

### 2.1 · Build index + filter to the 22 experiment classes

GastroVision has **27 classes** but is heavily long-tail. The paper (arXiv 2307.08140) keeps the
**22 classes with more than 5 images** and drops the 5 ultra-rare ones. We reproduce this **by the
same rule** (`count > 5`) instead of hardcoding names — robust to folder-name quirks — and then
**assert we end up with exactly 22**.

**The 22 classes (name — #images), verified from the paper's Table 3:**

- **Upper GI (9):** Barrett's esophagus (19), Blood in lumen (34), Duodenal bulb (41),
  Esophagitis (22), Gastric polyps (13), Gastroesophageal junction/normal z-line (66),
  Normal esophagus (28), Normal stomach (194), Pylorus (78)
- **Lower GI (13):** Accessory tools (253), Cecum (23), Colon diverticula (6), Colon polyps (163),
  Colorectal cancer (29), Dyed-lifted-polyps (28), Dyed-resection-margins (49), Ileocecal valve (40),
  Mucosal inflammation large bowel (6), Normal mucosa/vascular pattern (293), Resected polyps (18),
  Retroflex rectum (14), Small bowel/terminal ileum (169)

> Imbalance is extreme: **293 vs 6 images (~49×)**. This is exactly why the Data-70% work
> (class-balancing + macro-F1) matters, and why the baseline macro-F1 is only 0.6504.

In [ ]:
# ImageFolder needs a tree of <root>/<class_name>/*.jpg (exactly one level of class folders).
full = tv.datasets.ImageFolder(str(DATA_DIR))
ORIG_CLASSES = full.classes
all_samples = full.samples                       # list of (path, orig_class_idx)
all_labels = np.array([y for _, y in all_samples])

# Paper's exact rule: keep classes with MORE THAN 5 images -> the 22 experiment classes
# (5 ultra-rare classes dropped). Reproduces their selection without hardcoding names.
MIN_PER_CLASS = 6   # ">5 images" per arXiv 2307.08140
counts_orig = np.bincount(all_labels, minlength=len(ORIG_CLASSES))
keep = [c for c in range(len(ORIG_CLASSES)) if counts_orig[c] >= MIN_PER_CLASS]
remap = {c: i for i, c in enumerate(keep)}

CLASSES = [ORIG_CLASSES[c] for c in keep]
NUM_CLASSES = len(CLASSES)
samples = [(p, remap[y]) for (p, y) in all_samples if y in remap]
labels = np.array([y for _, y in samples])

dropped = [f'{ORIG_CLASSES[c]}({counts_orig[c]})'
           for c in range(len(ORIG_CLASSES)) if c not in remap]
print(f'kept {NUM_CLASSES}/{len(ORIG_CLASSES)} classes, {len(samples)} images')
if dropped:
    print(f'dropped (<={MIN_PER_CLASS - 1} imgs): {dropped}')

# Sanity check: the paper uses exactly 22 classes. If this fails, DATA_DIR most likely points
# one level too high (at Upper/Lower-GI category folders) or too low. Fix DATA_DIR, don't relax this.
assert NUM_CLASSES == 22, (
    f'Expected 22 classes (paper), got {NUM_CLASSES}. '
    f'Check DATA_DIR nesting — it must directly contain the class subfolders. Found: {ORIG_CLASSES}')

In [ ]:
# --- Fixed stratified 60:20:20 split (paper protocol). Keep EXACT across the team. ---
SPLIT_SEED = 42
idx = np.arange(len(samples))
train_idx, tmp_idx = train_test_split(
    idx, test_size=0.40, stratify=labels, random_state=SPLIT_SEED)
val_idx, test_idx = train_test_split(
    tmp_idx, test_size=0.50, stratify=labels[tmp_idx], random_state=SPLIT_SEED)
print(f'train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')

# NOTE: SPLIT_SEED is fixed and reported (rubric rule #1). The per-run SEEDS only affect
# model init / data ordering, NOT the split.

## 3 · EDA — long-tail check (Member A owns the deep dive)

This is the 20% "Data analysis" section. Below is a starter; expand with image-quality
checks, artifacts, suspicious labels, and a **leakage check**.

In [ ]:
import matplotlib.pyplot as plt
counts = pd.Series(labels).value_counts().sort_index()
counts.index = [CLASSES[i] for i in counts.index]
counts = counts.sort_values(ascending=False)
print('most/least common:')
print(counts.head(3)); print(counts.tail(3))
print(f'imbalance ratio (max/min): {counts.max() / counts.min():.1f}x')

plt.figure(figsize=(12, 4))
counts.plot(kind='bar'); plt.ylabel('# images')
plt.title('GastroVision class distribution (long-tail)')
plt.tight_layout(); plt.show()
# TODO (Member A): flag classes with few images -> macro-F1 will be noisy there.

## 4 · Datasets & transforms

> ⚠️ **Member A:** the augmentation here is a placeholder. Replace with **GI-domain-justified**
> transforms and prove each with an ablation (rubric section 3 = 30%).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),          # TODO(A): justify per GI domain
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GastroDataset(Dataset):
    def __init__(self, indices, transform):
        self.items = [samples[i] for i in indices]
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        img = full.loader(path).convert('RGB')   # some GI frames may be grayscale/RGBA
        return self.transform(img), y

def make_loaders():
    tr = DataLoader(GastroDataset(train_idx, train_tf), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    va = DataLoader(GastroDataset(val_idx, eval_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    te = DataLoader(GastroDataset(test_idx, eval_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return tr, va, te

## 5 · Shared evaluation harness (Member C owns; everyone imports)

**Primary metric = macro-F1** (rubric rule #3). Always report per-class F1 + confusion matrix.
`labels=range(NUM_CLASSES)` is passed everywhere so a rare class missing from predictions
never crashes the report.

In [ ]:
ALL_LABELS = list(range(NUM_CLASSES))

@torch.no_grad()
def evaluate(model, loader):
    """Return dict with macro_f1, micro_f1, y_true, y_pred."""
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    return {
        'macro_f1': f1_score(y_true, y_pred, labels=ALL_LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(y_true, y_pred, labels=ALL_LABELS, average='micro', zero_division=0),
        'y_true': y_true, 'y_pred': y_pred,
    }

def report_per_class(res):
    print(classification_report(res['y_true'], res['y_pred'], labels=ALL_LABELS,
          target_names=CLASSES, zero_division=0, digits=3))

def plot_confusion(res):
    import seaborn as sns
    cm = confusion_matrix(res['y_true'], res['y_pred'], labels=ALL_LABELS)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
    plt.xlabel('pred'); plt.ylabel('true'); plt.tight_layout(); plt.show()

## 6 · Baseline model — DenseNet-121 (reproduce first!)

**Rubric rule #5:** reproduce the strongest published baseline before claiming to beat it.
Target: land near the paper's macro-F1 **0.6504**. Report YOUR reproduced number.

In [ ]:
def build_densenet121(num_classes):
    m = tv.models.densenet121(weights=tv.models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m.to(DEVICE)

USE_AMP = (DEVICE == 'cuda')

def train_one(model, tr, va, epochs=EPOCHS, lr=1e-4, class_weights=None):
    """Minimal fine-tune loop w/ AMP + best-val checkpointing. Returns best val macro-F1."""
    crit = nn.CrossEntropyLoss(weight=class_weights)   # TODO(A): swap for LDAM/Balanced-Softmax
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        vf1 = evaluate(model, va)['macro_f1']
        if vf1 > best:
            best, best_state = vf1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'ep {ep+1:02d}/{epochs}  val_macroF1={vf1:.4f}  (best={best:.4f})')
    if best_state:
        model.load_state_dict(best_state)
    return best

In [ ]:
# --- Multi-seed runner: report mean +/- std on the TEST split (rubric rule #4). ---
def run_seeds(build_fn, seeds=SEEDS, tag='model', **train_kw):
    test_scores = []
    for s in seeds:
        set_seed(s)
        tr, va, te = make_loaders()
        model = build_fn(NUM_CLASSES)
        train_one(model, tr, va, **train_kw)
        res = evaluate(model, te)
        print(f'[seed {s}] test macro-F1 = {res["macro_f1"]:.4f}')
        test_scores.append(res['macro_f1'])
        torch.save(model.state_dict(), CKPT_DIR / f'{tag}_seed{s}.pt')   # survives disconnects
    mean, std = float(np.mean(test_scores)), float(np.std(test_scores))
    print(f'\n{tag} TEST macro-F1: {mean:.4f} +/- {std:.4f}')
    return {'tag': tag, 'scores': test_scores, 'mean': mean, 'std': std}

# Reproduce baseline: start with 1 seed to sanity-check ~0.65, then run all 3 seeds.
# baseline = run_seeds(build_densenet121, seeds=[0], tag='densenet121')   # quick check
# baseline = run_seeds(build_densenet121, tag='densenet121')              # full 3-seed

---
## 7 · Member work sections (branch from here)

Everyone reuses `run_seeds` / `evaluate` so results are comparable.

### 7A · Member A — Data (70%)
- GI-domain augmentation + **ablation table** (each step: on/off -> macro-F1).
- Imbalance: class weights -> LDAM / Balanced-Softmax / decoupled (cRT) / repeat-factor sampling.
- Leakage check + suspicious-label review.

### 7B · Member B — Model (30%): CNN vs Transformer vs Hybrid
- Reproduce DenseNet-121 (done above) + add ConvNeXt-T, Swin-T, ViT-S, CoAtNet via `timm`.
- Same protocol/seeds. Fill the extended baseline table (CNN vs Transformer vs Hybrid).

```python
# import timm
# def build_swin_t(num_classes):
#     m = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
#     return m.to(DEVICE)
# run_seeds(build_swin_t, tag='swin_t')
```

### 7C · Member C — Transfer learning + Deployment
- Linear probe vs progressive unfreezing vs layer-wise LR decay vs full fine-tune (comparison table).
- ONNX export + latency/size measurement + Gradio demo.

```python
# dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
# torch.onnx.export(model, dummy, str(OUTPUT_DIR / 'model.onnx'),
#                   input_names=['input'], output_names=['logits'], opset_version=17)
```